# Gesture Dataset Loading and LSTM Training
This notebook loads gesture datasets from the `custom` and `xml_logs` folders, compares their contents, and trains an LSTM model on the xml_logs dataset.  
This LSTM model is then evaluated with different hyperparameters and compared to the $1 unistroke recognizer.  

## Init

In [15]:
import os
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

np.random.seed(42)
tf.random.set_seed(42)

# Helper: Parse points from XML file
def parse_gesture_xml(file_path):
    xml_root = ET.parse(file_path).getroot()
    points = []
    for element in xml_root.findall("Point"):
        x = float(element.get("X"))
        y = float(element.get("Y"))
        points.append([x, y])
    return np.array(points, dtype=float), file_path

## 2. Load Custom and xml_logs Datasets
Load gesture data from both the `datasets/custom` and `datasets/xml_logs` directories. Parse XML files and load the data into pandas DataFrames or numpy arrays.

In [16]:
# Paths to datasets
datasets_dir = 'datasets'
custom_dir = os.path.join(datasets_dir, 'custom')
xml_logs_dir = os.path.join(datasets_dir, 'xml_logs')

# Helper: Recursively find all XML files in a directory
def find_xml_files(root_dir):
    return [y for x in os.walk(root_dir) for y in glob(os.path.join(x[0], '*.xml'))]

# Load custom dataset
custom_files = find_xml_files(custom_dir)
custom_data = []
custom_labels = []
for file in custom_files:
    points, path = parse_gesture_xml(file)
    label = os.path.basename(file).split('.')[0][:-2]
    custom_data.append(points)
    custom_labels.append(label)

# Load xml_logs dataset
xml_logs_files = find_xml_files(xml_logs_dir)
xml_logs_data = []
xml_logs_labels = []
for file in xml_logs_files:
    points, path = parse_gesture_xml(file)
    label = os.path.basename(file).split('.')[0][:-2]
    xml_logs_data.append(points)
    xml_logs_labels.append(label)

print(f"Loaded {len(custom_data)} samples from custom, {len(xml_logs_data)} from xml_logs.")

Loaded 52 samples from custom, 5280 from xml_logs.


## 3. Limit Dataset Size to Smallest Dataset
To ensure a fair comparison, limit both datasets to the size of the smaller one.

In [17]:
# Truncate both datasets so that for each label, the number of samples is the minimum count for that label across both datasets


# Count occurrences of each label in both datasets
custom_counts = Counter(custom_labels)
xml_logs_counts = Counter(xml_logs_labels)

# Find common labels
common_labels = set(custom_counts.keys()) & set(xml_logs_counts.keys())

# For each label, keep only the minimum number of samples in both datasets
def filter_to_min_samples(data, labels, min_counts):
    filtered_data = []
    filtered_labels = []
    label_counter = defaultdict(int)
    for d, l in zip(data, labels):
        if l in min_counts and label_counter[l] < min_counts[l]:
            filtered_data.append(d)
            filtered_labels.append(l)
            label_counter[l] += 1
    return filtered_data, filtered_labels

min_counts = {label: min(custom_counts[label], xml_logs_counts[label]) for label in common_labels}

custom_data_sym, custom_labels_sym = filter_to_min_samples(custom_data, custom_labels, min_counts)
xml_logs_data_sym, xml_logs_labels_sym = filter_to_min_samples(xml_logs_data, xml_logs_labels, min_counts)

print(f"Symmetrical datasets created with {sum(min_counts.values())} samples per dataset.")
custom_data = custom_data_sym
custom_labels = custom_labels_sym
xml_logs_data = xml_logs_data_sym
xml_logs_labels = xml_logs_labels_sym

Symmetrical datasets created with 52 samples per dataset.


## 4. Display Head of Both Datasets
Print the first few samples of both the custom and xml_logs datasets to inspect their structure.

In [18]:
# Show the first 3 samples of each dataset as DataFrames for inspection
def sample_to_df(points, label):
    df = pd.DataFrame(points, columns=['X', 'Y'])
    df['Label'] = label
    return df

print("Custom dataset head:")
for i in range(3):
    display(sample_to_df(custom_data[i], custom_labels[i]).head())
    print()

print("xml_logs dataset head:")
for i in range(3):
    display(sample_to_df(xml_logs_data[i], xml_logs_labels[i]).head())
    print()

Custom dataset head:


,X,Y,Label
0,513.0,306.0,check
1,513.0,307.0,check
2,514.0,308.0,check
3,517.0,311.0,check
4,520.0,313.0,check


,X,Y,Label
0,509.0,306.0,check
1,510.0,309.0,check
2,512.0,312.0,check
3,517.0,317.0,check
4,524.0,324.0,check


,X,Y,Label
0,475.0,227.0,check
1,477.0,228.0,check
2,481.0,229.0,check
3,484.0,232.0,check
4,489.0,236.0,check



xml_logs dataset head:


,X,Y,Label
0,86.0,195.0,check
1,88.0,199.0,check
2,89.0,201.0,check
3,91.0,205.0,check
4,95.0,209.0,check


,X,Y,Label
0,85.0,192.0,check
1,86.0,194.0,check
2,88.0,196.0,check
3,90.0,200.0,check
4,93.0,203.0,check


,X,Y,Label
0,91.0,190.0,check
1,92.0,193.0,check
2,94.0,197.0,check
3,97.0,200.0,check
4,99.0,203.0,check


## 5. Load Full xml_logs Dataset
Load the entire xml_logs dataset (not truncated) for LSTM training.

In [19]:
# Reload the entire xml_logs dataset for LSTM training
xml_logs_files_full = find_xml_files(xml_logs_dir)
xml_logs_data_full = []
xml_logs_labels_full = []
for file in xml_logs_files_full:
    points, path = parse_gesture_xml(file)
    label = os.path.basename(file).split('.')[0][:-2]
    xml_logs_data_full.append(points)
    xml_logs_labels_full.append(label)

print(f"Loaded {len(xml_logs_data_full)} samples from full xml_logs dataset.")

Loaded 5280 samples from full xml_logs dataset.


## 6. Preprocess Data for LSTM Training
Normalize and reshape the gesture data as required for LSTM input. Encode labels and split the data into training and validation sets.

In [20]:
# max sequence length
max_len = 64

def normalize_points(points, target_len=64):
    # Resample to fixed number of points
    points = np.array(points)
    if len(points) == 0:
        return np.zeros((target_len, 2))
    if len(points) == target_len:
        return points
    # Linear interpolation for resampling
    idxs = np.linspace(0, len(points) - 1, target_len)
    resampled = np.vstack([
        np.interp(idxs, np.arange(len(points)), points[:, 0]),
        np.interp(idxs, np.arange(len(points)), points[:, 1])
    ]).T
    return resampled

X = np.array([normalize_points(p, max_len) for p in xml_logs_data_full])
le = LabelEncoder()
y = le.fit_transform(xml_logs_labels_full)
y_cat = to_categorical(y)

# Split into train/val
X_train, X_val, y_train, y_val = train_test_split(X, y_cat, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (4224, 64, 2), y_train shape: (4224, 16)


## 7. Build and Train LSTM Model
Define and train an LSTM model using TensorFlow/Keras on the preprocessed xml_logs dataset.

In [21]:
# Define LSTM model
num_classes = y_cat.shape[1]
num_lstm_layers = 1
def get_layer_set(return_sequences=False, num_layers=1):
    if num_layers <= 0:
        return []
    return [
        LSTM(32, return_sequences=(num_layers > 1 or return_sequences)),
        Dropout(0.3),
        *get_layer_set(return_sequences=return_sequences, num_layers=num_layers - 1)
    ]


model = Sequential([
    Input(shape=(max_len, 2)),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    *get_layer_set(return_sequences=False, num_layers=num_lstm_layers),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping]
)

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_8 (LSTM)                   │ (None, 64, 64)         │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 16)             │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,096 (117.56 KB)

 Trainable params: 30,096 (117.56 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.1925 - loss: 2.4735 - val_accuracy: 0.6042 - val_loss: 1.3069
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5606 - loss: 1.3006 - val_accuracy: 0.8040 - val_loss: 0.7816
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7411 - loss: 0.8360 - val_accuracy: 0.8788 - val_loss: 0.4779
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8153 - loss: 0.6071 - val_accuracy: 0.8835 - val_loss: 0.4096
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8560 - loss: 0.4811 - val_accuracy: 0.9072 - val_loss: 0.3086
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8682 - loss: 0.4454 - val_accuracy: 0.9252 - val_loss: 0.2652
Epoch 7/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8895 - loss: 0.3726 - val_accuracy: 0.9176 - val_loss: 0.2704
Epoch 8/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8845 - loss: 0.3530 - val_accu

## 8. Evaluate Model Performance
Evaluate the trained LSTM model on the validation set and print relevant metrics.

In [22]:
# Evaluate the model on the validation set
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Loss: 0.0809
Validation Accuracy: 0.9782
